In [1]:
!pip install pdfplumber nltk spacy scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 96.0 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [2]:
import pandas as pd
import nltk
import pdfplumber
import zipfile
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
from google.colab import files

uploaded = files.upload()

Saving archive (2).zip to archive (2).zip


In [4]:
zip_path = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("Resume_Dataset")

    print("Dataset extracted successfully!")

Dataset extracted successfully!


In [5]:
df = pd.read_csv("Resume_Dataset/Resume/Resume.csv")

print("Dataset Loaded Successfully!")
print(df.head())

Dataset Loaded Successfully!
         ID                                         Resume_str  \
0  16852973           HR ADMINISTRATOR/MARKETING ASSOCIATE\...   
1  22323967           HR SPECIALIST, US HR OPERATIONS      ...   
2  33176873           HR DIRECTOR       Summary      Over 2...   
3  27018550           HR SPECIALIST       Summary    Dedica...   
4  17812897           HR MANAGER         Skill Highlights  ...   

                                         Resume_html Category  
0  <div class="fontsize fontface vmargins hmargin...       HR  
1  <div class="fontsize fontface vmargins hmargin...       HR  
2  <div class="fontsize fontface vmargins hmargin...       HR  
3  <div class="fontsize fontface vmargins hmargin...       HR  
4  <div class="fontsize fontface vmargins hmargin...       HR  


In [6]:
print(df.columns)

Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='object')


In [7]:
resume_text = str(df["Resume_str"][0])

print(resume_text[:1000])

         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss preve

In [8]:
job_description = """
We are looking for a Python Developer with skills in
Python,
SQL,
Machine Learning,
NLP,
Pandas,
Scikit-learn.
"""

documents = [resume_text, job_description]

tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(documents)

score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]

print(f"Resume Match Score : {score*100:.2f}%")

Resume Match Score : 1.37%


In [9]:
scores = []

for i in range(len(df)):
    resume = str(df["Resume_str"][i])

    docs = [resume, job_description]

    tfidf = TfidfVectorizer(stop_words="english")
    matrix = tfidf.fit_transform(docs)

    score = cosine_similarity(matrix[0:1], matrix[1:2])[0][0]

    scores.append((i, df["Category"][i], score))

scores = sorted(scores, key=lambda x: x[2], reverse=True)

print("===== TOP 10 MATCHING RESUMES =====")

for rank, item in enumerate(scores[:10], start=1):
    print(f"{rank}. Resume {item[0]} | Category: {item[1]} | Score: {item[2]*100:.2f}%")

===== TOP 10 MATCHING RESUMES =====
1. Resume 926 | Category: AGRICULTURE | Score: 10.36%
2. Resume 1348 | Category: AUTOMOBILE | Score: 9.06%
3. Resume 2450 | Category: AVIATION | Score: 8.24%
4. Resume 1339 | Category: AUTOMOBILE | Score: 7.88%
5. Resume 1762 | Category: ENGINEERING | Score: 7.72%
6. Resume 1756 | Category: ENGINEERING | Score: 7.48%
7. Resume 470 | Category: ADVOCATE | Score: 7.47%
8. Resume 2153 | Category: BANKING | Score: 7.34%
9. Resume 194 | Category: DESIGNER | Score: 7.30%
10. Resume 2291 | Category: ARTS | Score: 7.03%


In [10]:
required_skills = [
        "python",
            "sql",
                "machine learning",
                    "nlp",
                        "pandas",
                            "scikit-learn"
]

resume = resume_text.lower()

found = []
missing = []

for skill in required_skills:
    if skill.lower() in resume:
        found.append(skill)
    else:
        missing.append(skill)

print("Skills Found:")
print(found)

print("\nMissing Skills:")
print(missing)

Skills Found:
[]

Missing Skills:
['python', 'sql', 'machine learning', 'nlp', 'pandas', 'scikit-learn']


In [12]:
print("===== Resume Screening Report =====")

print(f"Resume Match Score : {score*100:.2f}%")
print(f"Skills Found       : {len(found)}")
print(f"Missing Skills     : {len(missing)}")

if score*100 >= 70:
    status = "Selected"
elif score*100 >= 40:
    status = "Review"
else:
    status = "Rejected"

print(f"Candidate Status   : {status}")

===== Resume Screening Report =====
Resume Match Score : 1.65%
Skills Found       : 0
Missing Skills     : 6
Candidate Status   : Rejected


In [13]:
top_score = scores[0][2] * 100

print("===== Resume Analysis =====")
print(f"Best Resume Score : {top_score:.2f}%")

if top_score >= 70:
    print("Reason: Strong match with the job description.")
elif top_score >= 40:
    print("Reason: Partial match.")
else:
    print("Reason: Low match. Important skills are missing.")

    print("\nMissing Skills:")

    for skill in missing:
        print("-", skill)

===== Resume Analysis =====
Best Resume Score : 10.36%
Reason: Low match. Important skills are missing.

Missing Skills:
- python
- sql
- machine learning
- nlp
- pandas
- scikit-learn


In [15]:
print("===== Candidate Ranking Explanation === ==")

for rank, item in enumerate(scores[:5], start=1):
    print(f"\nRank {rank}")
    print(f"Category : {item[1]}")
    print(f"Score    : {item[2]*100:.2f}%")
    print("Reason   : Higher similarity with the job description.")

===== Candidate Ranking Explanation === ==

Rank 1
Category : AGRICULTURE
Score    : 10.36%
Reason   : Higher similarity with the job description.

Rank 2
Category : AUTOMOBILE
Score    : 9.06%
Reason   : Higher similarity with the job description.

Rank 3
Category : AVIATION
Score    : 8.24%
Reason   : Higher similarity with the job description.

Rank 4
Category : AUTOMOBILE
Score    : 7.88%
Reason   : Higher similarity with the job description.

Rank 5
Category : ENGINEERING
Score    : 7.72%
Reason   : Higher similarity with the job description.


In [16]:
!pip freeze > requirements.txt

print("requirements.txt generated successfully.")

requirements.txt generated successfully.


In [17]:
from google.colab import files
files.download("requirements.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>